# Dimensional Magnitude Analysis
## Domain: Categorical Scale & Volume Distribution
### Objectives & Methodology:
- Compare key measure magnitudes (customer headcounts, revenue, unit volumes, and average costs) across categories.
- Assess geographic concentration of customers and item shipments across global territories.
- Analyze gender balance in the customer base.
- Evaluate product category catalog depth and corresponding average acquisition costs.
- Measure total revenue contributions generated across product categories and customer accounts.


In [ ]:
# Databricks notebook source
# Setup environment and display helper
import os
from pyspark.sql import SparkSession

try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)


## 1. Customer Distribution: Country & Gender Breakdown
Evaluate customer concentration across geographic territories and gender demographics.


In [ ]:
# Total customers by country
df_cust_by_country = spark.sql("""
SELECT
    country,
    COUNT(customer_key) AS total_customers
FROM gold.dim_customers
GROUP BY country
ORDER BY total_customers DESC
""")
display(df_cust_by_country)


In [ ]:
# Total customers by gender
df_cust_by_gender = spark.sql("""
SELECT
    gender,
    COUNT(customer_key) AS total_customers
FROM gold.dim_customers
GROUP BY gender
ORDER BY total_customers DESC
""")
display(df_cust_by_gender)


## 2. Product Catalog Distribution & Cost Profiling by Category
Analyze catalog diversity and average product cost across merchandising categories.


In [ ]:
# Total products by category
df_prod_by_category = spark.sql("""
SELECT
    category,
    COUNT(product_key) AS total_products
FROM gold.dim_products
GROUP BY category
ORDER BY total_products DESC
""")
display(df_prod_by_category)


In [ ]:
# Average product cost by category
df_avg_cost_category = spark.sql("""
SELECT
    category,
    AVG(cost) AS avg_cost
FROM gold.dim_products
GROUP BY category
ORDER BY avg_cost DESC
""")
display(df_avg_cost_category)


## 3. Revenue Distribution by Product Category and Customer
Measure top-line financial contribution broken down by product category and customer accounts.


In [ ]:
# Total revenue generated by product category
df_revenue_category = spark.sql("""
SELECT
    p.category,
    SUM(f.sales_amount) AS total_revenue
FROM gold.fact_sales f
LEFT JOIN gold.dim_products p ON p.product_key = f.product_key
GROUP BY p.category
ORDER BY total_revenue DESC
""")
display(df_revenue_category)


In [ ]:
# Total revenue generated by each customer
df_revenue_customer = spark.sql("""
SELECT
    c.customer_key,
    c.first_name,
    c.last_name,
    SUM(f.sales_amount) AS total_revenue
FROM gold.fact_sales f
LEFT JOIN gold.dim_customers c ON c.customer_key = f.customer_key
GROUP BY c.customer_key, c.first_name, c.last_name
ORDER BY total_revenue DESC
""")
display(df_revenue_customer)


## 4. Volume Distribution: Physical Units Sold by Country
Analyze physical supply chain distribution by quantifying items sold per country.


In [ ]:
# Distribution of items sold across countries
df_units_by_country = spark.sql("""
SELECT
    c.country,
    SUM(f.quantity) AS total_sold_items
FROM gold.fact_sales f
LEFT JOIN gold.dim_customers c ON c.customer_key = f.customer_key
GROUP BY c.country
ORDER BY total_sold_items DESC
""")
display(df_units_by_country)


## Business Takeaways & Magnitude Insights
- **Geographic Concentration**: The United States and Australia lead customer volume and physical product shipments.
- **Bikes Category Dominance**: Bikes command the highest average product cost and generate the overwhelming share of enterprise revenue.
- **Demographic Parity**: Balanced male and female customer counts indicate broad appeal across core product categories.
